In [2]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Install & authenticate with HuggingFace (WRITE access)
# ═══════════════════════════════════════════════════════
!pip install -q huggingface_hub

import os, json
from pathlib import Path
from huggingface_hub import HfApi, login

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise ValueError(
        "HF_TOKEN secret not found. Add a WRITE-access HuggingFace token "
        "as a Kaggle Secret named 'HF_TOKEN' via Add-ons → Secrets."
    )

login(token=hf_token)
api = HfApi()

# Confirm the token actually has write access before uploading anything
whoami = api.whoami(token=hf_token)
print(f"Authenticated as: {whoami['name']}")
print(f"Token permissions include write: "
      f"{'write' in str(whoami.get('auth', {}).get('accessToken', {}).get('role', '')).lower() or 'unknown — verify at huggingface.co/settings/tokens'}")

Authenticated as: Neha12210
Token permissions include write: True


In [3]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Verify all source paths BEFORE uploading
# (fail fast here rather than mid-upload)
# ═══════════════════════════════════════════════════════
YOLO_SRC       = Path("/kaggle/input/models/nehamalik10/yolo-detector/pytorch/default/1/yolo_best.pt")
TROCR_SRC      = Path("/kaggle/input/models/nehamalik10/trocr-base-finetuned-best/pytorch/default/1/trocr base finetune 3.pt")
ARTIFACTS_SRC  = Path("/kaggle/input/datasets/nehamalik10/phase4-session2-outputs/session2_artifacts.json")

for name, path in [("YOLO detector", YOLO_SRC), ("TrOCR checkpoint", TROCR_SRC), ("Vocab artifacts", ARTIFACTS_SRC)]:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at: {path}\n"
                                 f"Verify this notebook has the correct input attached.")
    size_mb = path.stat().st_size / 1e6
    print(f"✓ {name}: {path} ({size_mb:.1f} MB)")

# Sanity-check the artifacts JSON has the exact structure loader.py depends on
with open(ARTIFACTS_SRC, "r", encoding="utf-8") as f:
    art = json.load(f)

required_keys = {"char_to_token", "token_to_char", "VOCAB_SIZE", "PAD_ID", "BOS_ID", "EOS_ID", "UNK_ID"}
missing_keys = required_keys - set(art.keys())
if missing_keys:
    raise ValueError(f"session2_artifacts.json is missing required keys: {missing_keys}")

print(f"\n✓ Artifacts structure verified. VOCAB_SIZE={art['VOCAB_SIZE']}, "
      f"PAD_ID={art['PAD_ID']}, BOS_ID={art['BOS_ID']}, EOS_ID={art['EOS_ID']}, UNK_ID={art['UNK_ID']}")

✓ YOLO detector: /kaggle/input/models/nehamalik10/yolo-detector/pytorch/default/1/yolo_best.pt (22.5 MB)
✓ TrOCR checkpoint: /kaggle/input/models/nehamalik10/trocr-base-finetuned-best/pytorch/default/1/trocr base finetune 3.pt (1130.6 MB)
✓ Vocab artifacts: /kaggle/input/datasets/nehamalik10/phase4-session2-outputs/session2_artifacts.json (0.0 MB)

✓ Artifacts structure verified. VOCAB_SIZE=140, PAD_ID=0, BOS_ID=1, EOS_ID=2, UNK_ID=3


In [6]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Upload YOLO detector
# ═══════════════════════════════════════════════════════
YOLO_REPO = "Neha12210/hindi-htr-yolo"   # replace nehamalik10 with your actual HF username if different

api.create_repo(YOLO_REPO, exist_ok=True, repo_type="model")

api.upload_file(
    path_or_fileobj=str(YOLO_SRC),
    path_in_repo="yolo_best.pt",
    repo_id=YOLO_REPO,
    repo_type="model",
)
print(f"✅ YOLO uploaded to {YOLO_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ YOLO uploaded to Neha12210/hindi-htr-yolo


In [7]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Upload TrOCR checkpoint + session2_artifacts.json
# (uploading the ACTUAL artifacts file, not a placeholder
# char_vocab.json — this is the Fix 6 correction from the plan)
# ═══════════════════════════════════════════════════════
TROCR_REPO = "Neha12210/hindi-htr-trocr"

api.create_repo(TROCR_REPO, exist_ok=True, repo_type="model")

print("Uploading TrOCR checkpoint (~1.13GB, this will take a few minutes)...")
api.upload_file(
    path_or_fileobj=str(TROCR_SRC),
    path_in_repo="trocr_base_finetune_3.pt",
    repo_id=TROCR_REPO,
    repo_type="model",
)
print("✅ TrOCR checkpoint uploaded")

api.upload_file(
    path_or_fileobj=str(ARTIFACTS_SRC),
    path_in_repo="session2_artifacts.json",
    repo_id=TROCR_REPO,
    repo_type="model",
)
print("✅ Vocab artifacts uploaded")

Uploading TrOCR checkpoint (~1.13GB, this will take a few minutes)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ TrOCR checkpoint uploaded
✅ Vocab artifacts uploaded


In [8]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Verify: download each uploaded file back and confirm
# it matches the source (catches truncated/corrupted uploads
# before you find out the hard way during Modal deployment)
# ═══════════════════════════════════════════════════════
import hashlib
from huggingface_hub import hf_hub_download

def file_hash(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

verification_pairs = [
    (YOLO_SRC, YOLO_REPO, "yolo_best.pt"),
    (TROCR_SRC, TROCR_REPO, "trocr_base_finetune_3.pt"),
    (ARTIFACTS_SRC, TROCR_REPO, "session2_artifacts.json"),
]

all_verified = True
for src_path, repo_id, filename in verification_pairs:
    print(f"\nVerifying {filename}...")
    downloaded_path = hf_hub_download(repo_id=repo_id, filename=filename, force_download=True)
    src_hash = file_hash(src_path)
    dl_hash = file_hash(downloaded_path)
    match = src_hash == dl_hash
    all_verified = all_verified and match
    print(f"  Source hash:     {src_hash[:16]}...")
    print(f"  Downloaded hash: {dl_hash[:16]}...")
    print(f"  {'✅ MATCH' if match else '❌ MISMATCH — re-upload this file'}")

print(f"\n{'='*60}")
print(f"{'✅ ALL FILES VERIFIED — safe to proceed to Modal deployment' if all_verified else '❌ SOME FILES FAILED VERIFICATION — do not proceed, re-upload the mismatched file(s) above'}")
print(f"{'='*60}")


Verifying yolo_best.pt...


yolo_best.pt:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

  Source hash:     55faa5e7e684d0b2...
  Downloaded hash: 55faa5e7e684d0b2...
  ✅ MATCH

Verifying trocr_base_finetune_3.pt...


trocr_base_finetune_3.pt:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

  Source hash:     92710afdc0694219...
  Downloaded hash: 92710afdc0694219...
  ✅ MATCH

Verifying session2_artifacts.json...


session2_artifacts.json: 0.00B [00:00, ?B/s]

  Source hash:     95d80adad892d676...
  Downloaded hash: 95d80adad892d676...
  ✅ MATCH

✅ ALL FILES VERIFIED — safe to proceed to Modal deployment


In [10]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Print final repo URLs for reference
# ═══════════════════════════════════════════════════════
print("Model repositories ready for Modal deployment:\n")
print(f"YOLO:  https://huggingface.co/{YOLO_REPO}")
print(f"TrOCR: https://huggingface.co/{TROCR_REPO}")
print(f"TTS:   https://huggingface.co/ai4bharat/indic-parler-tts  (existing, no upload needed)")

Model repositories ready for Modal deployment:

YOLO:  https://huggingface.co/Neha12210/hindi-htr-yolo
TrOCR: https://huggingface.co/Neha12210/hindi-htr-trocr
TTS:   https://huggingface.co/ai4bharat/indic-parler-tts  (existing, no upload needed)
